In [156]:
# ==============================================================================
# 1. IMPORTS
# ==============================================================================

# OpenPIV
from openpiv import windef
from openpiv import tools, scaling, validation, filters, preprocess
import openpiv.pyprocess as process
from openpiv import pyprocess

# General
import numpy as np
import tifffile as tif
import napari
import matplotlib.pyplot as plt
import pandas as pd
from roifile import ImagejRoi

import pathlib
from time import time
import warnings
%matplotlib qt

# Custom functions
from src.PIV import run_PIV_on_frames

In [157]:
# ==============================================================================
# Paths
# ==============================================================================

unstressed_reference_path = '/mnt/crunch/Clark/Fly_TFM/data/second/unstressed_reference/'
sitting_reference_path = '/mnt/crunch/Clark/Fly_TFM/data/second/sitting_reference/'
time_resolved_path = '/mnt/crunch/Clark/Fly_TFM/data/second/time_resolved/'

In [158]:
# ==============================================================================
# Do subtraction using sitting reference
# ==============================================================================

starting_frame = 1
num_frames = 28
num_gridpoints = 3969 # len(sitting_reference)

u_sitting = np.zeros((num_frames, num_gridpoints))
v_sitting = np.zeros((num_frames, num_gridpoints))
    
for index, current_frame in enumerate(range(starting_frame, num_frames + 1)):
        
    # Load data
    data = pd.read_csv(sitting_reference_path + f'PIVlab_{current_frame:04d}.txt', skiprows=2)
        
    # Load columns and inject columns into subtaction method
    u_sitting[index] = data['u [px/frame]'].values #- sitting_reference['u [px/frame]'].values
    v_sitting[index] = data['v [px/frame]'].values #- sitting_reference['v [px/frame]'].values

In [159]:
# ==============================================================================
# Do subtraction ACTUALLY using sitting reference WRT THE UNSTRESSED REFERENCE
# ==============================================================================

sitting_reference = pd.read_csv('/mnt/crunch/Clark/Fly_TFM/data/second/unstressed_reference/PIVlab_0040.txt', skiprows=2)

starting_frame = 40
num_frames = 28
num_gridpoints = len(sitting_reference)

u_subtraction = np.zeros((num_frames, num_gridpoints))
v_subtraction = np.zeros((num_frames, num_gridpoints))
    
for index, current_frame in enumerate(range(starting_frame, starting_frame + num_frames + 0)):
        
    # Load data
    data = pd.read_csv(unstressed_reference_path + f'PIVlab_{current_frame:04d}.txt', skiprows=2)
        
    # Load columns and inject columns into subtaction method
    u_subtraction[index] = data['u [px/frame]'].values - sitting_reference['u [px/frame]'].values
    v_subtraction[index] = data['v [px/frame]'].values - sitting_reference['v [px/frame]'].values
    
    print(f'Frame {current_frame} loaded and subtracted from sitting reference.')
    print(f'Progress: {index + 1}/{num_frames} frames processed.')
    print(unstressed_reference_path + f'PIVlab_{current_frame:04d}.txt')

Frame 40 loaded and subtracted from sitting reference.
Progress: 1/28 frames processed.
/mnt/crunch/Clark/Fly_TFM/data/second/unstressed_reference/PIVlab_0040.txt
Frame 41 loaded and subtracted from sitting reference.
Progress: 2/28 frames processed.
/mnt/crunch/Clark/Fly_TFM/data/second/unstressed_reference/PIVlab_0041.txt
Frame 42 loaded and subtracted from sitting reference.
Progress: 3/28 frames processed.
/mnt/crunch/Clark/Fly_TFM/data/second/unstressed_reference/PIVlab_0042.txt
Frame 43 loaded and subtracted from sitting reference.
Progress: 4/28 frames processed.
/mnt/crunch/Clark/Fly_TFM/data/second/unstressed_reference/PIVlab_0043.txt
Frame 44 loaded and subtracted from sitting reference.
Progress: 5/28 frames processed.
/mnt/crunch/Clark/Fly_TFM/data/second/unstressed_reference/PIVlab_0044.txt
Frame 45 loaded and subtracted from sitting reference.
Progress: 6/28 frames processed.
/mnt/crunch/Clark/Fly_TFM/data/second/unstressed_reference/PIVlab_0045.txt
Frame 46 loaded and su

In [160]:
# ==============================================================================
# Do frame to frame subtraction using the unstressed reference
# ==============================================================================

wave_start = pd.read_csv('/mnt/crunch/Clark/Fly_TFM/data/second/unstressed_reference/PIVlab_0040.txt', skiprows=2)

starting_frame = 41
num_frames = 28
num_gridpoints = len(wave_start)

u_subtraction_frame_to_frame_unstressed = np.zeros((num_frames - 1, num_gridpoints))
v_subtraction_frame_to_frame_unstressed = np.zeros((num_frames - 1, num_gridpoints))

for index, current_frame in enumerate(range(starting_frame, starting_frame + num_frames - 1)):
    # Load data
    previous_data = pd.read_csv(unstressed_reference_path + f'PIVlab_{(current_frame - 1):04d}.txt', skiprows=2)
    current_data = pd.read_csv(unstressed_reference_path + f'PIVlab_{current_frame:04d}.txt', skiprows=2)
    
    u_subtraction_frame_to_frame_unstressed[index] = current_data['u [px/frame]'].values - previous_data['u [px/frame]'].values
    v_subtraction_frame_to_frame_unstressed[index] = current_data['v [px/frame]'].values - previous_data['v [px/frame]'].values
    
    print(f'Frame {current_frame} loaded and subtracted from previous frame.')
    print(f'Progress: {index + 1}/{num_frames} frames processed.')
    print(unstressed_reference_path + f'PIVlab_{(current_frame - 1):04d}.txt')
    print(unstressed_reference_path + f'PIVlab_{current_frame:04d}.txt')
    
print(u_subtraction_frame_to_frame_unstressed.shape)

Frame 41 loaded and subtracted from previous frame.
Progress: 1/28 frames processed.
/mnt/crunch/Clark/Fly_TFM/data/second/unstressed_reference/PIVlab_0040.txt
/mnt/crunch/Clark/Fly_TFM/data/second/unstressed_reference/PIVlab_0041.txt
Frame 42 loaded and subtracted from previous frame.
Progress: 2/28 frames processed.
/mnt/crunch/Clark/Fly_TFM/data/second/unstressed_reference/PIVlab_0041.txt
/mnt/crunch/Clark/Fly_TFM/data/second/unstressed_reference/PIVlab_0042.txt
Frame 43 loaded and subtracted from previous frame.
Progress: 3/28 frames processed.
/mnt/crunch/Clark/Fly_TFM/data/second/unstressed_reference/PIVlab_0042.txt
/mnt/crunch/Clark/Fly_TFM/data/second/unstressed_reference/PIVlab_0043.txt
Frame 44 loaded and subtracted from previous frame.
Progress: 4/28 frames processed.
/mnt/crunch/Clark/Fly_TFM/data/second/unstressed_reference/PIVlab_0043.txt
/mnt/crunch/Clark/Fly_TFM/data/second/unstressed_reference/PIVlab_0044.txt
Frame 45 loaded and subtracted from previous frame.
Progress

In [161]:
# integrate previous cell

# u_time_resolved_subtraction_frame_to_frame_unstressed = np.zeros((num_frames - 1, num_gridpoints))
# v_time_resolved_subtraction_frame_to_frame_unstressed = np.zeros((num_frames - 1, num_gridpoints))

# for index, current_frame in enumerate(range(1, num_frames)):
    
#     u_time_resolved_subtraction_frame_to_frame_unstressed[index] = 

u_integrated_subtraction_frame_to_frame_unstressed = np.cumsum(u_subtraction_frame_to_frame_unstressed, axis=0)
v_integrated_subtraction_frame_to_frame_unstressed = np.cumsum(v_subtraction_frame_to_frame_unstressed, axis=0)

In [162]:
# ==============================================================================
# Integrate starting from sitting reference
# ==============================================================================

#NOTE Remember, need to add a frame manually before every other frame for visualization of integrated displacements

u_time_resolved = np.zeros((num_frames - 1, num_gridpoints))
v_time_resolved = np.zeros((num_frames - 1, num_gridpoints))

# Load .csv columns into numpy arrays
for index, current_frame in enumerate(range(1, num_frames)):
    
    # Load data
    data = pd.read_csv(time_resolved_path + f'PIVlab_{current_frame:04d}.txt', skiprows=2)
    
    # Load columns of currently integrated displacement
    u_time_resolved[index] = data['u [px/frame]'].values
    v_time_resolved[index] = data['v [px/frame]'].values
    
# Integrate
u_integrated = np.cumsum(u_time_resolved, axis=0)
v_integrated = np.cumsum(v_time_resolved, axis=0)

print(u_integrated.shape)

(27, 3969)


In [163]:
%%script true
# ==============================================================================
# Visualize the subtraction method
# ==============================================================================

u_grid = u_subtraction.reshape(28, 63, 63)
v_grid = v_subtraction.reshape(28, 63, 63)
z_grid = np.zeros_like(u_grid) # napari expects num spatial dimensions = num vector components, so manually set dz = 0

vector_grid = np.stack([z_grid, v_grid, u_grid], axis=-1)
magnitude_grid = np.sqrt(u_grid**2 + v_grid**2)

# ------------------------------------------------------------------------------

viewer = napari.Viewer()

magnitude_layer = viewer.add_image(
    magnitude_grid,
    name='Magnitude Heatmap',
    colormap='jet',
    #interpolation2d='bicubic',
    contrast_limits=[0, 3]
)

vector_layer = viewer.add_vectors(
    vector_grid,
    name="Displacements",
    vector_style='arrow',
    edge_width=0.2,
    length=1.0,
    edge_color="black"
)

magnitude_layer.colorbar.visible = True

napari.run()

In [164]:
%%script true
# ==============================================================================
# Visualize the integration method
# ==============================================================================

u_grid = u_integrated.reshape(27, 63, 63)
v_grid = v_integrated.reshape(27, 63, 63)
z_grid = np.zeros_like(u_grid)

vector_grid = np.stack([z_grid, v_grid, u_grid], axis=-1)
magnitude_grid = np.sqrt(u_grid**2 + v_grid**2)

# ------------------------------------------------------------------------------

viewer = napari.Viewer()

magnitude_layer = viewer.add_image(
    magnitude_grid,
    name='Magnitude Heatmap',
    colormap='jet',
    #interpolation2d='bicubic',
    contrast_limits=[0, 3]
)

vector_layer = viewer.add_vectors(
    vector_grid,
    name="Displacements",
    vector_style='arrow',
    edge_width=0.2,
    length=1.0,
    edge_color="black"
)

magnitude_layer.colorbar.visible = True

napari.run()

In [165]:
%%script true
# ==============================================================================
# Visualize the residual
# ==============================================================================

# Add a new frame to the integrated displacement arrays to match the number of frames in the subtraction method
new_row = np.zeros((1, num_gridpoints))
u_integrated_from_zero = np.vstack((new_row, u_integrated))
v_integrated_from_zero = np.vstack((new_row, v_integrated))

u_residual = u_integrated_from_zero - u_subtraction
v_residual = v_integrated_from_zero - v_subtraction

u_grid = u_residual.reshape(28, 63, 63)
v_grid = v_residual.reshape(28, 63, 63)
z_grid = np.zeros_like(u_grid)

vector_grid = np.stack([z_grid, v_grid, u_grid], axis=-1)
magnitude_grid = np.sqrt(u_grid**2 + v_grid**2)

# ------------------------------------------------------------------------------

viewer = napari.Viewer()

magnitude_layer = viewer.add_image(
    magnitude_grid,
    name='Residual Heatmap',
    colormap='jet',
    #interpolation2d='bicubic',
    contrast_limits=[0, 3]
)

vector_layer = viewer.add_vectors(
    vector_grid,
    name="Residual Displacements",
    vector_style='arrow',
    edge_width=0.2,
    length=1.0,
    edge_color="black"
)

magnitude_layer.colorbar.visible = True

napari.run()

In [166]:
%%script true
# ==============================================================================
# 1. Prepare Integration Data (27 frames -> shifted to 28 frames)
# ==============================================================================
u_grid_integrated_raw = u_integrated.reshape(27, 63, 63)
v_grid_integrated_raw = v_integrated.reshape(27, 63, 63)
z_grid_integrated_raw = np.zeros_like(u_grid_integrated_raw)

# Prepend 1 frame of zeros to the front (axis=0) so frame 0 is empty
u_grid_integrated = np.pad(u_grid_integrated_raw, ((1, 0), (0, 0), (0, 0)), mode='constant', constant_values=0)
v_grid_integrated = np.pad(v_grid_integrated_raw, ((1, 0), (0, 0), (0, 0)), mode='constant', constant_values=0)
z_grid_integrated = np.pad(z_grid_integrated_raw, ((1, 0), (0, 0), (0, 0)), mode='constant', constant_values=0)

vector_grid_integrated = np.stack([z_grid_integrated, u_grid_integrated, v_grid_integrated], axis=-1)
magnitude_grid_integrated = np.sqrt(u_grid_integrated**2 + v_grid_integrated**2)

# ==============================================================================
# 2. Prepare Subtraction Data (28 frames)
# ==============================================================================
u_grid_subtraction = u_subtraction.reshape(28, 63, 63)
v_grid_subtraction = v_subtraction.reshape(28, 63, 63)
z_grid_subtraction = np.zeros_like(u_grid_subtraction)

vector_grid_subtraction = np.stack([z_grid_subtraction, u_grid_subtraction, v_grid_subtraction], axis=-1)
magnitude_grid_subtraction = np.sqrt(u_grid_subtraction**2 + v_grid_subtraction**2)

# RESIDUAL
# Add a new frame to the integrated displacement arrays to match the number of frames in the subtraction method
new_row = np.zeros((1, num_gridpoints))
u_integrated_from_zero = np.vstack((new_row, u_integrated))
v_integrated_from_zero = np.vstack((new_row, v_integrated))

u_residual_integrated_vs_subtraction = u_integrated_from_zero - u_subtraction
v_residual_integrated_vs_subtraction = v_integrated_from_zero - v_subtraction

u_grid_residual = u_residual_integrated_vs_subtraction.reshape(28, 63, 63)
v_grid_residual = v_residual_integrated_vs_subtraction.reshape(28, 63, 63)
z_grid_residual = np.zeros_like(u_grid_residual)

vector_grid_residual = np.stack([z_grid_residual, u_grid_residual, v_grid_residual], axis=-1)
magnitude_grid_residual = np.sqrt(u_grid_residual**2 + v_grid_residual**2)

# ==============================================================================
# 3. Launch single viewer with both datasets
# ==============================================================================
magnitude_grid_sitting = np.sqrt(u_sitting.reshape(28, 63, 63)**2 + v_sitting.reshape(28, 63, 63)**2)
vector_grid_sitting = np.stack([np.zeros_like(u_sitting.reshape(28, 63, 63)), u_sitting.reshape(28, 63, 63), v_sitting.reshape(28, 63, 63)], axis=-1)

viewer = napari.Viewer()

contrast_limits = [0, 3]

# --- Sitting Reference Layers ---
mag_sitting_layer = viewer.add_image(
    magnitude_grid_sitting,
    name='Sitting Reference - Heatmap',
    colormap='jet',
    contrast_limits=contrast_limits
)

vec_sitting_layer = viewer.add_vectors(
    vector_grid_sitting,
    name='Sitting Reference - Vectors',
    vector_style='arrow',
    edge_width=0.2,
    length=1.0,
    edge_color='black'
)

# --- Integration Method Layers ---
mag_int_layer = viewer.add_image(
    magnitude_grid_integrated,
    name='Integration - Heatmap',
    colormap='jet',
    contrast_limits=contrast_limits
)

vec_int_layer = viewer.add_vectors(
    vector_grid_integrated,
    name='Integration - Vectors',
    vector_style='arrow',
    edge_width=0.2,
    length=1.0,
    edge_color='black'
)

# --- Subtraction Method Layers ---
mag_sub_layer = viewer.add_image(
    magnitude_grid_subtraction,
    name='Subtraction - Heatmap',
    colormap='jet',
    contrast_limits=contrast_limits
)

vec_sub_layer = viewer.add_vectors(
    vector_grid_subtraction,
    name='Subtraction - Vectors',
    vector_style='arrow',
    edge_width=0.2,
    length=1.0,
    edge_color='black'
)

# --- Residual (Integrated vs Subtraction) Method Layers ---
mag_res_layer = viewer.add_image(
    magnitude_grid_residual,
    name='Residual (Integrated vs Subtraction) - Heatmap',
    colormap='jet',
    contrast_limits=contrast_limits
)

vec_res_layer = viewer.add_vectors(
    vector_grid_residual,
    name='Residual (Integrated vs Subtraction) - Vectors',
    vector_style='arrow',
    edge_width=0.2,
    length=1.0,
    edge_color='black'
)

# --- Residual (Integrated vs Sitting) Method Layers ---
u_residual_integrated_vs_sitting = u_integrated_from_zero - u_sitting
v_residual_integrated_vs_sitting = v_integrated_from_zero - v_sitting

magnitude_grid_residual_sitting = np.sqrt(u_residual_integrated_vs_sitting.reshape(28, 63, 63)**2 + v_residual_integrated_vs_sitting.reshape(28, 63, 63)**2)
vector_grid_residual_sitting = np.stack([np.zeros_like(u_residual_integrated_vs_sitting.reshape(28, 63, 63)), u_residual_integrated_vs_sitting.reshape(28, 63, 63), v_residual_integrated_vs_sitting.reshape(28, 63, 63)], axis=-1)

mag_res_sitting_layer = viewer.add_image(
    magnitude_grid_residual_sitting,
    name='Residual (Integrated vs Sitting) - Heatmap',
    colormap='jet',
    contrast_limits=contrast_limits
)

vec_res_sitting_layer = viewer.add_vectors(
    vector_grid_residual_sitting,
    name='Residual (Integrated vs Sitting) - Vectors',
    vector_style='arrow',
    edge_width=0.2,
    length=1.0,
    edge_color='black'
)

# ------------------------------------------------------------------------------

# Colorbar display
mag_sub_layer.colorbar.visible = True

# Side-by-side grid view (2 layers per box)
viewer.grid.enabled = True
viewer.grid.stride = 2

napari.run()

In [167]:
# Pad integration arrays (27 frames -> 28) so everything lines up
new_row = np.zeros((1, num_gridpoints))
u_integration = np.vstack((new_row, u_integrated))
v_integration = np.vstack((new_row, v_integrated))

u_integrated_subtraction_frame_to_frame_unstressed = np.vstack((new_row, u_integrated_subtraction_frame_to_frame_unstressed))
v_integrated_subtraction_frame_to_frame_unstressed = np.vstack((new_row, v_integrated_subtraction_frame_to_frame_unstressed))

# Residuals
u_res_int_vs_sub = u_integration - u_subtraction
v_res_int_vs_sub = v_integration - v_subtraction

u_res_int_vs_sitting = u_integration - u_sitting
v_res_int_vs_sitting = v_integration - v_sitting

u_residual_integration_vs_integration_subtraction_frame_to_frame = u_integrated_subtraction_frame_to_frame_unstressed - u_integration
v_residual_integration_vs_integration_subtraction_frame_to_frame = v_integrated_subtraction_frame_to_frame_unstressed - v_integration

datasets = [
    #("Sitting Reference",                  u_sitting,     v_sitting),
    ("Integration Method",                 u_integration, v_integration),
    ("Integration Method (Subtraction frame to frame)", u_integrated_subtraction_frame_to_frame_unstressed, v_integrated_subtraction_frame_to_frame_unstressed),
    #("Subtraction Method",                 u_subtraction, v_subtraction),
    #("Residual (Integration - Subtraction)", u_res_int_vs_sub,     v_res_int_vs_sub),
    #("Residual (Integration - Sitting)",     u_res_int_vs_sitting, v_res_int_vs_sitting),
    ("Residual (Integration - Integration Subtraction Frame to Frame)", u_residual_integration_vs_integration_subtraction_frame_to_frame, v_residual_integration_vs_integration_subtraction_frame_to_frame)
]

# ------------------------------------------------------------------------------

viewer = napari.Viewer()

colormap='jet'
contrast_limits=[0, 3]

arrow_style = 'arrow'
arrow_edge_color = 'black'
arrow_width = 0.2
arrow_length = 1.0

scale_factor = 2048/63

for label, u, v in datasets:
    u_grid = u.reshape(28, 63, 63)
    v_grid = v.reshape(28, 63, 63)
    z_grid = np.zeros_like(u_grid)
    
    #u_grid = np.rot90(-u_grid, k=1, axes=(1, 2))  # Rotate the u and v grids to match the correct orientation
    #v_grid = np.rot90(v_grid, k=1, axes=(1, 2))  # Rotate the u and v grids to match the correct orientation
    
    #u_grid = np.flip(u_grid, axis=1)  # Flip the u and v grids to match the correct orientation
    #v_grid = -np.flip(v_grid, axis=1)  # Flip the u and
    
    vector_grid = np.stack([z_grid, u_grid, v_grid], axis=-1)
    magnitude_grid = np.sqrt(u_grid**2 + v_grid**2)

    viewer.add_image(magnitude_grid, name=f'{label} - Heatmap', colormap=colormap, contrast_limits=contrast_limits, scale=(1, scale_factor, scale_factor))
    viewer.add_vectors(vector_grid, name=f'{label} - Vectors', vector_style=arrow_style,
                        edge_width=arrow_width, length=arrow_length, edge_color=arrow_edge_color, scale=(1, scale_factor, scale_factor))


# Add actual image
one_wave = tif.imread('/mnt/crunch/Clark/Fly_TFM/data/second/second_best_one_wave.tif')
viewer.add_image(one_wave, name='Actual Image', colormap='gray', contrast_limits=[0, 255])

# Colorbar display
viewer.layers['Integration Method - Heatmap'].colorbar.visible = True

# Side-by-side grid view (2 layers per box)
viewer.grid.enabled = True
viewer.grid.stride = 2

napari.run()